In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

src = V("csv")
chk = f"{V('checkpoints')}/bronze_ais"   # el MISMO checkpoint de 02_bronze
tbl = T("bronze", "ais_raw")

def show(df, n=50):
    print(df.limit(n).toPandas().to_string(index=False))

def conteo():
    return spark.table(tbl).count()

primer_csv = sorted(f.path for f in dbutils.fs.ls(src) if f.name.endswith(".csv"))[0]
with open(primer_csv.replace("dbfs:", "")) as fh:
    header = fh.readline().strip().split(",")
schema_csv = StructType([StructField(c, StringType(), True) for c in header])

def iniciar_stream(archivo_falla=None):
    df = (spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("cloudFiles.maxFilesPerTrigger", 1)
          .option("header", "true")
          .option("rescuedDataColumn", "_rescued_data")
          .schema(schema_csv)
          .load(src)
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingested_at", F.current_timestamp()))
    if archivo_falla:   # falla inyectada: el batch que contenga este archivo aborta sin commit
        df = df.filter(F.when(F.col("_source_file").contains(archivo_falla),
                              F.raise_error(F.lit(f"FALLA SIMULADA en {archivo_falla}")))
                        .otherwise(F.lit(True)))
    return (df.writeStream
            .option("checkpointLocation", chk)
            .trigger(availableNow=True)
            .toTable(tbl))

In [0]:
base = conteo()
print(f"antes de la falla: {base:,}")

q = iniciar_stream(archivo_falla="AIS_2023_06_07")
try:
    q.awaitTermination()
except Exception as e:
    print("STREAM FALLÓ:", str(e)[:150])

tras_falla = conteo()
print(f"después de la falla: {tras_falla:,} | nuevas: {tras_falla - base:,}")
show(spark.table(tbl).groupBy("_source_file").count().orderBy("_source_file"))

In [0]:
q = iniciar_stream()
q.awaitTermination()

final = conteo()
print(f"después de recuperar: {final:,} | nuevas en la recuperación: {final - tras_falla:,}")

In [0]:
crudo = (spark.read.text(f"{src}/*.csv")
         .select(F.regexp_replace(F.col("_metadata.file_path"), "^dbfs:", "").alias("_source_file"))
         .groupBy("_source_file").count()
         .withColumn("lineas_csv", F.col("count") - 1).drop("count"))

conc = (spark.table(tbl).groupBy("_source_file").count()
        .withColumnRenamed("count", "filas_bronze")
        .join(crudo, "_source_file", "full")
        .withColumn("diferencia", F.col("filas_bronze") - F.col("lineas_csv"))
        .orderBy("_source_file"))
show(conc)